This was our original notebook before we switched over to python scripts. Used to populate the database after running setup.

In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sqlite3
import requests
import time
from datetime import datetime
from pathlib import Path


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "setup").exists() and (candidate / "apiserver").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
DB_PATH = PROJECT_ROOT / "setup" / "travel_planner.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# Connect to the database used by the Flask API.
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Safety check
cursor.execute("SELECT COUNT(*) FROM Destinations")
dest_count = cursor.fetchone()[0]

# Check a second table to ensure the full script finished last time
cursor.execute("SELECT COUNT(*) FROM Weather_Monthly")
weather_count = cursor.fetchone()[0]

# Only exit if both tables have data
if dest_count > 0 and weather_count > 0:
    print(f"Database already fully populated. Skipping.")
    conn.close()
    sys.exit(0) # type: ignore

# If we are here, something was missing or empty, so we clear and start fresh
print("Database empty or incomplete. Resetting and re-populating...")
cursor.execute("DELETE FROM Destinations")
cursor.execute("DELETE FROM Weather_Monthly")
conn.commit()


# Dictionary to hold realistic cost and landmark data 

city_metadata = {
    "Tokyo": {"cost": "Expensive", "landmark": "Senso-ji Temple"},
    "Delhi": {"cost": "Medium", "landmark": "the Red Fort"},
    "Shanghai": {"cost": "Medium", "landmark": "The Bund"},
    "São Paulo": {"cost": "Medium", "landmark": "Paulista Avenue"},
    "Mexico City": {"cost": "Cheap", "landmark": "the Zócalo"},
    "Cairo": {"cost": "Cheap", "landmark": "the Pyramids of Giza"},
    "Mumbai": {"cost": "Medium", "landmark": "the Gateway of India"},
    "Beijing": {"cost": "Medium", "landmark": "the Forbidden City"},
    "Osaka": {"cost": "Expensive", "landmark": "Osaka Castle"},
    "Karachi": {"cost": "Cheap", "landmark": "Mazar-e-Quaid"},
    "Lagos": {"cost": "Cheap", "landmark": "Lekki Conservation Centre"},
    "Istanbul": {"cost": "Medium", "landmark": "Hagia Sophia"},
    "Buenos Aires": {"cost": "Cheap", "landmark": "Casa Rosada"},
    "Kolkata": {"cost": "Cheap", "landmark": "the Victoria Memorial"},
    "Manila": {"cost": "Cheap", "landmark": "Intramuros"},
    "Guangzhou": {"cost": "Medium", "landmark": "Canton Tower"},
    "Rio de Janeiro": {"cost": "Medium", "landmark": "Christ the Redeemer"},
    "Bogotá": {"cost": "Cheap", "landmark": "Mount Monserrate"},
    "Lima": {"cost": "Cheap", "landmark": "Huaca Pucllana"},
    "Bangkok": {"cost": "Cheap", "landmark": "the Grand Palace"},
    "Jakarta": {"cost": "Cheap", "landmark": "the National Monument"},
    "London": {"cost": "Expensive", "landmark": "Big Ben"},
    "New York": {"cost": "Expensive", "landmark": "the Statue of Liberty"},
    "Paris": {"cost": "Expensive", "landmark": "the Eiffel Tower"},
    "Tehran": {"cost": "Cheap", "landmark": "Golestan Palace"},
    "Hong Kong": {"cost": "Expensive", "landmark": "Victoria Peak"},
    "Hong Kong Island": {"cost": "Expensive", "landmark": "Victoria Peak"},
    "Taipei": {"cost": "Medium", "landmark": "Taipei 101"},
    "Riyadh": {"cost": "Expensive", "landmark": "the Kingdom Centre"},
    "Miami": {"cost": "Expensive", "landmark": "South Beach"},
    "Toronto": {"cost": "Expensive", "landmark": "the CN Tower"},
    "Sydney": {"cost": "Expensive", "landmark": "the Sydney Opera House"},
    "Melbourne": {"cost": "Expensive", "landmark": "Federation Square"},
    "Berlin": {"cost": "Medium", "landmark": "the Brandenburg Gate"},
    "Rome": {"cost": "Expensive", "landmark": "the Colosseum"},
    "Madrid": {"cost": "Medium", "landmark": "the Royal Palace"},
    "Seoul": {"cost": "Medium", "landmark": "Gyeongbokgung Palace"},
    "Los Angeles": {"cost": "Expensive", "landmark": "the Hollywood Walk of Fame"},
    "Chicago": {"cost": "Expensive", "landmark": "Millennium Park"},
    "Singapore": {"cost": "Expensive", "landmark": "Marina Bay Sands"},
    "Dubai": {"cost": "Expensive", "landmark": "the Burj Khalifa"},
    "Moscow": {"cost": "Medium", "landmark": "Red Square"},
    "Kuala Lumpur": {"cost": "Medium", "landmark": "the Petronas Twin Towers"},
    "Santiago": {"cost": "Medium", "landmark": "San Cristobal Hill"},
    "Johannesburg": {"cost": "Medium", "landmark": "the Apartheid Museum"},
    "Nairobi": {"cost": "Cheap", "landmark": "Nairobi National Park"},
    "Athens": {"cost": "Medium", "landmark": "the Acropolis"},
    "Vienna": {"cost": "Expensive", "landmark": "Schönbrunn Palace"},
    "Amsterdam": {"cost": "Expensive", "landmark": "the Anne Frank House"},
    "Warsaw": {"cost": "Cheap", "landmark": "the Old Town Market Square"},
    "Budapest": {"cost": "Cheap", "landmark": "the Parliament Building"},
    "Prague": {"cost": "Medium", "landmark": "the Charles Bridge"},
    "Stockholm": {"cost": "Expensive", "landmark": "the Vasa Museum"},
    "Brussels": {"cost": "Expensive", "landmark": "the Grand Place"},
    "Lisbon": {"cost": "Medium", "landmark": "Belém Tower"},
    "Dublin": {"cost": "Expensive", "landmark": "the Guinness Storehouse"},
    "Havana": {"cost": "Cheap", "landmark": "Old Havana"},
    "Caracas": {"cost": "Cheap", "landmark": "Avila National Park"},
    "Cape Town": {"cost": "Medium", "landmark": "Table Mountain"},
    "Auckland": {"cost": "Expensive", "landmark": "the Sky Tower"},
    "Casablanca": {"cost": "Medium", "landmark": "Hassan II Mosque"}
}


# Dictionary mapping realistic activities to cities

city_activities = {
    "Tokyo": ["Eating local food", "Shopping", "Nightlife and partying", "Theme parks and family fun", "Visiting historical sites"],
    "Delhi": ["Eating local food", "Visiting historical sites", "Exploring street markets"],
    "Shanghai": ["Shopping", "Nightlife and partying", "Eating local food", "Taking boat rides"],
    "São Paulo": ["Nightlife and partying", "Eating local food", "Watching live shows or sports"],
    "Mexico City": ["Eating local food", "Visiting historical sites", "Exploring street markets"],
    "Cairo": ["Visiting historical sites", "Exploring street markets"],
    "Mumbai": ["Eating local food", "Shopping", "Visiting historical sites", "Taking boat rides"],
    "Beijing": ["Visiting historical sites", "Eating local food", "Exploring street markets"],
    "Osaka": ["Eating local food", "Theme parks and family fun", "Nightlife and partying", "Shopping"],
    "Karachi": ["Eating local food", "Exploring street markets", "Visiting historical sites"],
    "Lagos": ["Nightlife and partying", "Exploring street markets", "Going to the beach"],
    "Istanbul": ["Visiting historical sites", "Exploring street markets", "Taking boat rides", "Eating local food"],
    "Buenos Aires": ["Eating local food", "Nightlife and partying", "Visiting historical sites"],
    "Kolkata": ["Eating local food", "Visiting historical sites", "Exploring street markets"],
    "Manila": ["Shopping", "Nightlife and partying", "Eating local food"],
    "Guangzhou": ["Eating local food", "Shopping", "Taking boat rides"],
    "Rio de Janeiro": ["Going to the beach", "Nightlife and partying", "Hiking and nature walks", "Eating local food"],
    "Bogotá": ["Hiking and nature walks", "Nightlife and partying", "Visiting historical sites"],
    "Lima": ["Eating local food", "Visiting historical sites", "Going to the beach"],
    "Bangkok": ["Eating local food", "Shopping", "Nightlife and partying", "Visiting historical sites", "Exploring street markets"],
    "Jakarta": ["Shopping", "Eating local food", "Nightlife and partying"],
    "London": ["Visiting historical sites", "Watching live shows or sports", "Shopping", "Eating local food", "Taking boat rides"],
    "New York": ["Watching live shows or sports", "Shopping", "Eating local food", "Nightlife and partying", "Visiting historical sites"],
    "Paris": ["Visiting historical sites", "Eating local food", "Shopping", "Taking boat rides"],
    "Tehran": ["Visiting historical sites", "Eating local food", "Exploring street markets"],
    "Hong Kong": ["Shopping", "Eating local food", "Taking boat rides", "Hiking and nature walks"],
    "Hong Kong Island": ["Shopping", "Eating local food", "Hiking and nature walks"],
    "Taipei": ["Eating local food", "Shopping", "Exploring street markets"],
    "Riyadh": ["Shopping", "Eating local food", "Visiting historical sites"],
    "Miami": ["Going to the beach", "Nightlife and partying", "Taking boat rides"],
    "Toronto": ["Eating local food", "Watching live shows or sports", "Shopping"],
    "Sydney": ["Going to the beach", "Taking boat rides", "Eating local food", "Watching live shows or sports"],
    "Melbourne": ["Eating local food", "Watching live shows or sports", "Shopping"],
    "Berlin": ["Nightlife and partying", "Visiting historical sites", "Eating local food"],
    "Rome": ["Visiting historical sites", "Eating local food", "Exploring street markets"],
    "Madrid": ["Eating local food", "Nightlife and partying", "Visiting historical sites"],
    "Seoul": ["Shopping", "Eating local food", "Nightlife and partying", "Visiting historical sites"],
    "Los Angeles": ["Going to the beach", "Theme parks and family fun", "Nightlife and partying", "Shopping"],
    "Chicago": ["Eating local food", "Watching live shows or sports", "Visiting historical sites"],
    "Singapore": ["Shopping", "Eating local food", "Theme parks and family fun"],
    "Dubai": ["Shopping", "Going to the beach", "Theme parks and family fun", "Nightlife and partying"],
    "Moscow": ["Visiting historical sites", "Nightlife and partying", "Eating local food"],
    "Kuala Lumpur": ["Shopping", "Eating local food", "Nightlife and partying"],
    "Santiago": ["Hiking and nature walks", "Eating local food", "Visiting historical sites"],
    "Johannesburg": ["Visiting historical sites", "Eating local food", "Shopping"],
    "Nairobi": ["Hiking and nature walks", "Exploring street markets"],
    "Athens": ["Visiting historical sites", "Eating local food", "Exploring street markets"],
    "Vienna": ["Visiting historical sites", "Watching live shows or sports", "Eating local food"],
    "Amsterdam": ["Taking boat rides", "Nightlife and partying", "Visiting historical sites"],
    "Warsaw": ["Visiting historical sites", "Eating local food", "Nightlife and partying"],
    "Budapest": ["Nightlife and partying", "Visiting historical sites", "Eating local food"],
    "Prague": ["Visiting historical sites", "Nightlife and partying", "Eating local food"],
    "Stockholm": ["Taking boat rides", "Visiting historical sites", "Eating local food"],
    "Brussels": ["Eating local food", "Visiting historical sites", "Shopping"],
    "Lisbon": ["Visiting historical sites", "Eating local food", "Nightlife and partying"],
    "Dublin": ["Nightlife and partying", "Visiting historical sites", "Eating local food"],
    "Havana": ["Visiting historical sites", "Nightlife and partying", "Taking boat rides"],
    "Caracas": ["Hiking and nature walks", "Eating local food", "Visiting historical sites"],
    "Cape Town": ["Hiking and nature walks", "Going to the beach", "Eating local food", "Taking boat rides"],
    "Auckland": ["Taking boat rides", "Hiking and nature walks", "Eating local food"],
    "Casablanca": ["Visiting historical sites", "Eating local food", "Exploring street markets"]
}


# NEW: Dictionary mapping spiritual/energy vibes to cities

city_vibes = {
    "Tokyo": ["Urban Pulse", "High Energy", "Knowledge & Discovery"],
    "Delhi": ["Spiritual Awakening", "High Energy", "Nostalgia & Heritage"],
    "Shanghai": ["Urban Pulse", "High Energy", "Creative Inspiration"],
    "São Paulo": ["High Energy", "Creative Inspiration", "Urban Pulse"],
    "Mexico City": ["Nostalgia & Heritage", "Creative Inspiration", "High Energy"],
    "Cairo": ["Nostalgia & Heritage", "Knowledge & Discovery", "Spiritual Awakening"],
    "Mumbai": ["High Energy", "Spiritual Awakening", "Urban Pulse"],
    "Beijing": ["Knowledge & Discovery", "Nostalgia & Heritage", "Urban Pulse"],
    "Osaka": ["High Energy", "Urban Pulse", "Knowledge & Discovery"],
    "Karachi": ["High Energy", "Nostalgia & Heritage", "Urban Pulse"],
    "Lagos": ["High Energy", "Creative Inspiration", "Urban Pulse"],
    "Istanbul": ["Nostalgia & Heritage", "Romantic Charm", "Spiritual Awakening"],
    "Buenos Aires": ["Romantic Charm", "High Energy", "Creative Inspiration"],
    "Kolkata": ["Spiritual Awakening", "Creative Inspiration", "Nostalgia & Heritage"],
    "Manila": ["High Energy", "Urban Pulse", "Raw Adventure"],
    "Guangzhou": ["Urban Pulse", "Knowledge & Discovery"],
    "Rio de Janeiro": ["High Energy", "Raw Adventure", "Romantic Charm"],
    "Bogotá": ["Creative Inspiration", "High Energy", "Raw Adventure"],
    "Lima": ["Nostalgia & Heritage", "Raw Adventure", "Knowledge & Discovery"],
    "Bangkok": ["High Energy", "Spiritual Awakening", "Urban Pulse"],
    "Jakarta": ["Urban Pulse", "High Energy"],
    "London": ["Knowledge & Discovery", "Nostalgia & Heritage", "Creative Inspiration"],
    "New York": ["High Energy", "Urban Pulse", "Creative Inspiration"],
    "Paris": ["Romantic Charm", "Creative Inspiration", "Nostalgia & Heritage"],
    "Tehran": ["Nostalgia & Heritage", "Knowledge & Discovery"],
    "Hong Kong": ["Urban Pulse", "High Energy", "Raw Adventure"],
    "Hong Kong Island": ["Urban Pulse", "High Energy", "Raw Adventure"],
    "Taipei": ["Urban Pulse", "Knowledge & Discovery", "Healthy & Wellness"],
    "Riyadh": ["Urban Pulse", "Knowledge & Discovery"],
    "Miami": ["High Energy", "Healthy & Wellness", "Urban Pulse"],
    "Toronto": ["Creative Inspiration", "Urban Pulse", "Healthy & Wellness"],
    "Sydney": ["Healthy & Wellness", "Raw Adventure", "High Energy"],
    "Melbourne": ["Creative Inspiration", "Knowledge & Discovery", "Urban Pulse"],
    "Berlin": ["Creative Inspiration", "High Energy", "Urban Pulse"],
    "Rome": ["Nostalgia & Heritage", "Romantic Charm", "Knowledge & Discovery"],
    "Madrid": ["High Energy", "Romantic Charm", "Creative Inspiration"],
    "Seoul": ["Urban Pulse", "Knowledge & Discovery", "Creative Inspiration"],
    "Los Angeles": ["Creative Inspiration", "Healthy & Wellness", "High Energy"],
    "Chicago": ["Urban Pulse", "Creative Inspiration", "Nostalgia & Heritage"],
    "Singapore": ["Healthy & Wellness", "Urban Pulse", "Peace & Serenity"],
    "Dubai": ["Urban Pulse", "High Energy", "Knowledge & Discovery"],
    "Moscow": ["Nostalgia & Heritage", "Knowledge & Discovery"],
    "Kuala Lumpur": ["Urban Pulse", "High Energy"],
    "Santiago": ["Raw Adventure", "Creative Inspiration", "Peace & Serenity"],
    "Johannesburg": ["Raw Adventure", "High Energy", "Knowledge & Discovery"],
    "Nairobi": ["Raw Adventure", "High Energy", "Peace & Serenity"],
    "Athens": ["Nostalgia & Heritage", "Knowledge & Discovery", "Romantic Charm"],
    "Vienna": ["Romantic Charm", "Creative Inspiration", "Peace & Serenity"],
    "Amsterdam": ["Creative Inspiration", "Peace & Serenity", "Romantic Charm"],
    "Warsaw": ["Nostalgia & Heritage", "Creative Inspiration"],
    "Budapest": ["Romantic Charm", "Nostalgia & Heritage", "Peace & Serenity"],
    "Prague": ["Romantic Charm", "Nostalgia & Heritage", "Creative Inspiration"],
    "Stockholm": ["Peace & Serenity", "Healthy & Wellness", "Creative Inspiration"],
    "Brussels": ["Knowledge & Discovery", "Creative Inspiration"],
    "Lisbon": ["Romantic Charm", "Nostalgia & Heritage", "Peace & Serenity"],
    "Dublin": ["High Energy", "Nostalgia & Heritage", "Creative Inspiration"],
    "Havana": ["Nostalgia & Heritage", "High Energy", "Romantic Charm"],
    "Caracas": ["Raw Adventure", "High Energy"],
    "Cape Town": ["Raw Adventure", "Healthy & Wellness", "Peace & Serenity"],
    "Auckland": ["Raw Adventure", "Peace & Serenity", "Healthy & Wellness"],
    "Casablanca": ["Romantic Charm", "Nostalgia & Heritage"]
}


# ACTIVITIES

new_activities = [
    ("Eating local food",), ("Going to the beach",), ("Visiting historical sites",), 
    ("Shopping",), ("Nightlife and partying",), ("Hiking and nature walks",), 
    ("Theme parks and family fun",), ("Watching live shows or sports",), 
    ("Exploring street markets",), ("Taking boat rides",)
]
cursor.executemany("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)
conn.commit()

# Create a quick reference dictionary to get ActivityID by its Name
cursor.execute("SELECT ActivityName, ActivityID FROM Activities")
activity_map = {row[0]: row[1] for row in cursor.fetchall()}


# TRAVEL VIBES (Spiritual & Energy)

new_travel_vibes = [
    ("Healthy & Wellness",), ("High Energy",), ("Knowledge & Discovery",), 
    ("Peace & Serenity",), ("Creative Inspiration",), ("Spiritual Awakening",), 
    ("Romantic Charm",), ("Raw Adventure",), ("Nostalgia & Heritage",), 
    ("Urban Pulse",)
]
cursor.executemany("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibes)
conn.commit()

# Create a quick reference dictionary to get VibeID by its Name
cursor.execute("SELECT VibeName, VibeID FROM Travel_Vibes")
vibe_map = {row[0]: row[1] for row in cursor.fetchall()}


# AUTOMATICALLY FETCH 50 CITIES FROM AN OPEN API

print("Fetching major global cities from API...")

search_terms = [
    "Tokyo", "Delhi", "Shanghai", "Sao Paulo", "Mexico City", "Cairo", "Mumbai", 
    "Beijing", "Osaka", "Karachi", "Lagos", "Istanbul", "Buenos Aires", "Kolkata", 
    "Manila", "Guangzhou", "Rio de Janeiro", "Bogota", "Lima", "Bangkok", "Jakarta", 
    "London", "New York", "Paris", "Tehran", "Hong Kong", "Taipei", "Riyadh", "Miami", 
    "Toronto", "Sydney", "Melbourne", "Berlin", "Rome", "Madrid", "Seoul", "Los Angeles",
    "Chicago", "Singapore", "Dubai", "Moscow", "Kuala Lumpur", "Santiago", "Johannesburg",
    "Nairobi", "Athens", "Vienna", "Amsterdam", "Warsaw", "Budapest", "Prague", "Stockholm",
    "Brussels", "Lisbon", "Dublin", "Havana", "Caracas", "Cape Town", "Auckland", "Casablanca"
]

fallback_countries = {
    "Tokyo": "Japan", "Delhi": "India", "Shanghai": "China", "Sao Paulo": "Brazil",
    "Mexico City": "Mexico", "Cairo": "Egypt", "Mumbai": "India", "Beijing": "China",
    "Osaka": "Japan", "Karachi": "Pakistan", "Lagos": "Nigeria", "Istanbul": "Turkey",
    "Buenos Aires": "Argentina", "Kolkata": "India", "Manila": "Philippines",
    "Guangzhou": "China", "Rio de Janeiro": "Brazil", "Bogota": "Colombia",
    "Lima": "Peru", "Bangkok": "Thailand", "Jakarta": "Indonesia", "London": "United Kingdom",
    "New York": "United States", "Paris": "France", "Tehran": "Iran", "Hong Kong": "China",
    "Taipei": "Taiwan", "Riyadh": "Saudi Arabia", "Miami": "United States",
    "Toronto": "Canada", "Sydney": "Australia", "Melbourne": "Australia", "Berlin": "Germany",
    "Rome": "Italy", "Madrid": "Spain", "Seoul": "South Korea", "Los Angeles": "United States",
    "Chicago": "United States", "Singapore": "Singapore", "Dubai": "United Arab Emirates",
    "Moscow": "Russia", "Kuala Lumpur": "Malaysia", "Santiago": "Chile",
    "Johannesburg": "South Africa", "Nairobi": "Kenya", "Athens": "Greece", "Vienna": "Austria",
    "Amsterdam": "Netherlands", "Warsaw": "Poland", "Budapest": "Hungary", "Prague": "Czechia",
    "Stockholm": "Sweden", "Brussels": "Belgium", "Lisbon": "Portugal", "Dublin": "Ireland",
    "Havana": "Cuba", "Caracas": "Venezuela", "Cape Town": "South Africa",
    "Auckland": "New Zealand", "Casablanca": "Morocco",
}

def add_fallback_city(term):
    if len(cities_to_add) >= 50:
        return False

    city_name = term
    if any(existing_city[0] == city_name for existing_city in cities_to_add):
        return False

    country_name = fallback_countries.get(city_name, "Unknown Country")
    landmark = city_metadata.get(city_name, {}).get("landmark", "its famous city center")
    description = f"{city_name} is a major travel destination in {country_name}, with {landmark} as a highlight."
    cities_to_add.append((city_name, country_name, description))
    print(f"Fallback added {city_name} - Total: {len(cities_to_add)}/50")
    return True


cities_to_add = []

for term in search_terms:
    if len(cities_to_add) >= 50:
        break

    added_for_term = False
    api_url = f"https://geocoding-api.open-meteo.com/v1/search?name={term}&count=5&format=json"

    try:
        response = requests.get(api_url, timeout=15)
        data = response.json()
        
        if "results" in data:
            for item in data["results"]:
                city_name = item["name"]
                country_name = item.get("country", "Unknown Country")
                population = item.get("population", 0)
                
                if population >= 1000000:
                    landmark = city_metadata.get(city_name, {}).get("landmark", "its famous city center")
                    description = f"{city_name} has a population of {population:,} and a famous place to visit is {landmark}."
                    
                    is_duplicate = any(existing_city[0] == city_name for existing_city in cities_to_add)
                    
                    if not is_duplicate and len(cities_to_add) < 50:
                        city_tuple = (city_name, country_name, description)
                        cities_to_add.append(city_tuple)
                        added_for_term = True
                        print(f"Added {city_name} (Pop: {population:,}) - Total: {len(cities_to_add)}/50")
                        
    except Exception as e:
        print(f"Skipped live lookup for '{term}' due to network error.")

    if not added_for_term:
        add_fallback_city(term)

    time.sleep(0.2) 

cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)
conn.commit()

cursor.execute("SELECT DestinationID, CityName FROM Destinations")
saved_cities = cursor.fetchall()

def insert_fallback_weather(destination_id, city_name):
    seed = sum(ord(char) for char in city_name)
    base_temp = 10 + (seed % 18)
    seasonal_offsets = [-5, -3, 1, 5, 9, 12, 14, 13, 9, 5, 0, -3]

    for month in range(1, 13):
        avg_temp = round(base_temp + seasonal_offsets[month - 1], 1)
        rainfall = round(18 + ((seed + month * 11) % 85), 1)
        cursor.execute("""
            INSERT INTO Weather_Monthly (DestinationID, Month, AvgTempC, RainfallMM)
            VALUES (?, ?, ?, ?)
        """, (destination_id, month, avg_temp, rainfall))



# AUTOMATICALLY POPULATE COST, WEATHER, VIBES & ACTIVITIES


for city in saved_cities:
    dest_id = city[0]
    city_name = city[1]
    
    # --- Relational Assignment: Realistic Cost Profiles ---
    chosen_cost = city_metadata.get(city_name, {}).get("cost", "Medium")
    
    cursor.execute("""
        INSERT INTO Cost_Profiles (DestinationID, BudgetLevel)
        VALUES (?, ?)
    """, (dest_id, chosen_cost))
    
    # --- Relational Assignment: Activities Link Table ---
    city_activity_names = city_activities.get(city_name, ["Eating local food", "Visiting historical sites", "Shopping"])
    
    for act_name in city_activity_names:
        act_id = activity_map[act_name] 
        spotlight = f"Make sure to enjoy {act_name.lower()} while visiting {city_name}!"
        
        cursor.execute("""
            INSERT INTO Destinations_Activities (DestinationID, ActivityID, Spotlight_Description)
            VALUES (?, ?, ?)
        """, (dest_id, act_id, spotlight))
    
    # --- Relational Assignment: Vibes Link Table ---
    # Look up the city's specific vibes, or give it a generic default if missing
    city_vibe_names = city_vibes.get(city_name, ["Urban Pulse", "Knowledge & Discovery"])
    
    for vibe_name in city_vibe_names:
        v_id = vibe_map[vibe_name]
        cursor.execute("""
            INSERT INTO Destination_Vibes (DestinationID, VibeID)
            VALUES (?, ?)
        """, (dest_id, v_id))
    
    # --- Relational Assignment: Weather Logs ---
    print(f"Downloading 12-month climate tracking profile for {city_name}...")
    
    weather_rows_inserted = 0

    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
        geo_data = requests.get(geo_url, timeout=15).json()
        
        if "results" in geo_data and len(geo_data["results"]) > 0:
            lat = geo_data["results"][0]["latitude"]
            lon = geo_data["results"][0]["longitude"]
            
            archive_url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-01-01&end_date=2023-12-31&daily=temperature_2m_mean,precipitation_sum"
            weather_data = requests.get(archive_url, timeout=30).json()
            
            dates = weather_data["daily"]["time"]
            temps = weather_data["daily"]["temperature_2m_mean"]
            rains = weather_data["daily"]["precipitation_sum"]
            
            monthly_temps = {m: [] for m in range(1, 13)}
            monthly_rains = {m: [] for m in range(1, 13)}
            
            for index in range(len(dates)):
                d = dates[index]
                t = temps[index]
                r = rains[index]
                
                if t is not None:
                    if r is not None:
                        current_date = datetime.strptime(d, "%Y-%m-%d")
                        month_num = current_date.month
                        monthly_temps[month_num].append(t)
                        monthly_rains[month_num].append(r)
            
            for month in range(1, 13):
                temp_list = monthly_temps[month]
                rain_list = monthly_rains[month]
                
                if len(temp_list) > 0:
                    avg_temp = sum(temp_list) / len(temp_list)
                    total_rain = sum(rain_list)
                    
                    cursor.execute("""
                        INSERT INTO Weather_Monthly (DestinationID, Month, AvgTempC, RainfallMM)
                        VALUES (?, ?, ?, ?)
                    """, (dest_id, month, round(avg_temp, 1), round(total_rain, 1)))
                    weather_rows_inserted += 1
                    
    except Exception as e:
        print(f"Weather sync skipped for {city_name}; using generated fallback climate data.")

    if weather_rows_inserted == 0:
        insert_fallback_weather(dest_id, city_name)

    time.sleep(0.5) 

conn.commit()
print(f"\nSuccess! {DB_PATH} has been compiled and linked cleanly with {len(saved_cities)} cities.")
conn.close()

Database empty or incomplete. Resetting and re-populating...
Fetching major global cities from API...
Added Tokyo (Pop: 9,733,276) - Total: 1/50
Added Delhi (Pop: 11,034,555) - Total: 2/50
Added Shanghai (Pop: 24,874,500) - Total: 3/50
Added São Paulo (Pop: 12,400,232) - Total: 4/50
Added Mexico City (Pop: 12,294,193) - Total: 5/50
Added Cairo (Pop: 9,606,916) - Total: 6/50
Added Mumbai (Pop: 12,691,836) - Total: 7/50
Added Beijing (Pop: 18,960,744) - Total: 8/50
Added Osaka (Pop: 2,753,862) - Total: 9/50
Added Karachi (Pop: 11,624,219) - Total: 10/50
Added Lagos (Pop: 15,388,000) - Total: 11/50
Added Istanbul (Pop: 15,701,602) - Total: 12/50
Added Buenos Aires (Pop: 2,891,082) - Total: 13/50
Added Kolkata (Pop: 4,631,392) - Total: 14/50
Added Manila (Pop: 1,600,000) - Total: 15/50
Added Guangzhou (Pop: 16,096,724) - Total: 16/50
Added Rio de Janeiro (Pop: 6,747,815) - Total: 17/50
Added Bogotá (Pop: 7,674,366) - Total: 18/50
Added Lima (Pop: 7,737,002) - Total: 19/50
Added Bangkok (Po